# Tiny Shakespeare Dataset - TinyGPT From Scratch

### [Tiny Shakespeare Dataset](https://github.com/karpathy/char-rnn)

Author: [Kevin Thomas](mailto:ket189@pitt.edu)

License: MIT

## Citation

[1] Andrej Karpathy, https://github.com/karpathy/nanoGPT

[2] Ashish Vaswani et al., https://arxiv.org/abs/1706.03762

## Install Libraries

In [ ]:
# conda activate prod
# conda install -c conda-forge pytorch torchvision torchaudio
# conda install numpy matplotlib

## Import Libraries

In [ ]:
import urllib.request
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
%matplotlib inline

## Seed

In [ ]:
SEED = 42
SEED

In [ ]:
torch.manual_seed(SEED)

## Parameters

In [ ]:
BLOCK_SIZE = 64
BLOCK_SIZE

In [ ]:
EMBED_DIM = 64
EMBED_DIM

In [ ]:
N_HEADS = 4
N_HEADS

In [ ]:
N_LAYERS = 2
N_LAYERS

In [ ]:
FF_DIM = 128
FF_DIM

## Hyperparameters

In [ ]:
LEARNING_RATE = 0.002
LEARNING_RATE

In [ ]:
EPOCHS = 10
EPOCHS

In [ ]:
BATCH_SIZE = 128
BATCH_SIZE

## Device

In [ ]:
DEVICE = torch.device(
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu")
DEVICE

## Download Dataset

In [ ]:
URL = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
urllib.request.urlretrieve(URL, 'tinyshakespeare.txt')
with open('tinyshakespeare.txt', encoding='utf-8') as handle:
    text = handle.read()
print('length:', len(text))

## Build Vocabulary

In [ ]:
CHARS = sorted(set(text))
STOI = {ch: i for i, ch in enumerate(CHARS)}
ITOS = {i: ch for i, ch in enumerate(CHARS)}
VOCAB_SIZE = len(CHARS)
data = torch.tensor([STOI[ch] for ch in text], dtype=torch.long)
print('vocab size:', VOCAB_SIZE)

## Create Dataset and DataLoader

In [ ]:
class CharDataset(torch.utils.data.Dataset):
    """
    A dataset of fixed-length character windows and next-character targets.
    """

    def __init__(self, data, block_size):
        """
        Store the encoded text and window length.

        Parameters:
            data (torch.Tensor): Encoded text.
            block_size (int): Length of each input window.

        Returns:
            None
        """
        self.data = data
        self.block_size = block_size

    def __len__(self):
        """
        Return the number of windows.

        Parameters:
            None

        Returns:
            int: Number of windows.
        """
        return len(self.data) - self.block_size

    def __getitem__(self, idx):
        """
        Return one input window and its shifted target.

        Parameters:
            idx (int): Window index.

        Returns:
            tuple: Input and target tensors.
        """
        x = self.data[idx:idx + self.block_size]
        y = self.data[idx + 1:idx + self.block_size + 1]
        return x, y

In [ ]:
split = int(0.9 * len(data))
train_data = CharDataset(data[:split], BLOCK_SIZE)
val_data = CharDataset(data[split:], BLOCK_SIZE)
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_data, batch_size=BATCH_SIZE, shuffle=False)
len(train_loader), len(val_loader)

## Attention Head

In [ ]:
class Head(nn.Module):
    """
    A single head of causal self-attention.
    """

    def __init__(self, embed_dim, head_dim, block_size):
        """
        Initialize key, query, value, and the causal mask.

        Parameters:
            embed_dim (int): Input embedding size.
            head_dim (int): Size of this head.
            block_size (int): Maximum sequence length.

        Returns:
            None
        """
        super(Head, self).__init__()
        self.key = nn.Linear(embed_dim, head_dim, bias=False)
        self.query = nn.Linear(embed_dim, head_dim, bias=False)
        self.value = nn.Linear(embed_dim, head_dim, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        """
        Apply causal self-attention to the input.

        Parameters:
            x (torch.Tensor): Input of shape (batch, time, embed_dim).

        Returns:
            torch.Tensor: Attention output.
        """
        _, time, _ = x.shape
        key = self.key(x)
        query = self.query(x)
        scores = query @ key.transpose(-2, -1) * key.size(-1) ** -0.5
        scores = scores.masked_fill(self.tril[:time, :time] == 0, float('-inf'))
        weights = torch.softmax(scores, dim=-1)
        return weights @ self.value(x)

## Multi-Head Attention

In [ ]:
class MultiHeadAttention(nn.Module):
    """
    Several attention heads running in parallel.
    """

    def __init__(self, embed_dim, n_heads, block_size):
        """
        Initialize the heads and the output projection.

        Parameters:
            embed_dim (int): Input embedding size.
            n_heads (int): Number of heads.
            block_size (int): Maximum sequence length.

        Returns:
            None
        """
        super(MultiHeadAttention, self).__init__()
        head_dim = embed_dim // n_heads
        self.heads = nn.ModuleList(
            [Head(embed_dim, head_dim, block_size) for _ in range(n_heads)])
        self.proj = nn.Linear(embed_dim, embed_dim)

    def forward(self, x):
        """
        Concatenate the heads and project the result.

        Parameters:
            x (torch.Tensor): Input of shape (batch, time, embed_dim).

        Returns:
            torch.Tensor: Attention output.
        """
        out = torch.cat([head(x) for head in self.heads], dim=-1)
        return self.proj(out)

## Feed-Forward Network

In [ ]:
class FeedForward(nn.Module):
    """
    A position-wise feed-forward network.
    """

    def __init__(self, embed_dim, ff_dim):
        """
        Initialize the two linear layers and activation.

        Parameters:
            embed_dim (int): Input embedding size.
            ff_dim (int): Hidden size.

        Returns:
            None
        """
        super(FeedForward, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(embed_dim, ff_dim),
            nn.ReLU(),
            nn.Linear(ff_dim, embed_dim))

    def forward(self, x):
        """
        Run the feed-forward network.

        Parameters:
            x (torch.Tensor): Input tensor.

        Returns:
            torch.Tensor: Output tensor.
        """
        return self.net(x)

## Transformer Block

In [ ]:
class Block(nn.Module):
    """
    A transformer block with attention and a feed-forward network.
    """

    def __init__(self, embed_dim, n_heads, block_size, ff_dim):
        """
        Initialize attention, feed-forward, and normalization layers.

        Parameters:
            embed_dim (int): Input embedding size.
            n_heads (int): Number of attention heads.
            block_size (int): Maximum sequence length.
            ff_dim (int): Feed-forward hidden size.

        Returns:
            None
        """
        super(Block, self).__init__()
        self.attn = MultiHeadAttention(embed_dim, n_heads, block_size)
        self.ffwd = FeedForward(embed_dim, ff_dim)
        self.ln1 = nn.LayerNorm(embed_dim)
        self.ln2 = nn.LayerNorm(embed_dim)

    def forward(self, x):
        """
        Run the block with residual connections.

        Parameters:
            x (torch.Tensor): Input tensor.

        Returns:
            torch.Tensor: Output tensor.
        """
        x = x + self.attn(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

## Create TinyGPT

In [ ]:
class TinyGPT(nn.Module):
    """
    A small decoder-only transformer language model.
    """

    def __init__(self, vocab_size, embed_dim, n_heads, n_layers,
                 block_size, ff_dim):
        """
        Initialize embeddings, blocks, and the output head.

        Parameters:
            vocab_size (int): Number of characters.
            embed_dim (int): Embedding size.
            n_heads (int): Number of attention heads.
            n_layers (int): Number of transformer blocks.
            block_size (int): Maximum sequence length.
            ff_dim (int): Feed-forward hidden size.

        Returns:
            None
        """
        super(TinyGPT, self).__init__()
        self.block_size = block_size
        self.token_embed = nn.Embedding(vocab_size, embed_dim)
        self.pos_embed = nn.Embedding(block_size, embed_dim)
        self.blocks = nn.Sequential(*[
            Block(embed_dim, n_heads, block_size, ff_dim) for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, vocab_size)

    def embed(self, idx):
        """
        Combine token and positional embeddings.

        Parameters:
            idx (torch.Tensor): Input indices.

        Returns:
            torch.Tensor: Embedded input.
        """
        _, time = idx.shape
        positions = torch.arange(time, device=idx.device)
        return self.token_embed(idx) + self.pos_embed(positions)

    def forward(self, idx, targets=None):
        """
        Run the forward pass and optional loss.

        Parameters:
            idx (torch.Tensor): Input indices.
            targets (torch.Tensor): Optional target indices.

        Returns:
            tuple: Logits and optional loss.
        """
        x = self.blocks(self.embed(idx))
        logits = self.head(self.ln_f(x))
        if targets is None:
            return logits, None
        loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    def generate(self, idx, max_new_tokens):
        """
        Generate new tokens by sampling from the model.

        Parameters:
            idx (torch.Tensor): Seed indices.
            max_new_tokens (int): Number of tokens to generate.

        Returns:
            torch.Tensor: The extended indices.
        """
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]
            logits, _ = self(idx_cond)
            probs = F.softmax(logits[:, -1, :], dim=-1)
            idx_next = torch.multinomial(probs, 1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

## Instantiate Model and Optimizer

In [ ]:
torch.manual_seed(SEED)
model = TinyGPT(VOCAB_SIZE, EMBED_DIM, N_HEADS, N_LAYERS, BLOCK_SIZE, FF_DIM).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
sum(p.numel() for p in model.parameters())

## Train Model

### Functions

In [ ]:
def train_epoch(model, loader, optimizer):
    """
    Train the model for a single epoch.

    Parameters:
        model (nn.Module): The model to train.
        loader (DataLoader): Training data loader.
        optimizer (torch.optim.Optimizer): Parameter update rule.

    Returns:
        float: Mean training loss.
    """
    model.train()
    total = 0.0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        _, loss = model(x, y)
        loss.backward()
        optimizer.step()
        total += loss.item() * len(y)
    return total / len(loader.dataset)


def estimate_loss(model, loader):
    """
    Estimate the mean loss over a loader.

    Parameters:
        model (nn.Module): The model to evaluate.
        loader (DataLoader): Evaluation data loader.

    Returns:
        float: Mean loss.
    """
    model.eval()
    total = 0.0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            _, loss = model(x, y)
            total += loss.item() * len(y)
    return total / len(loader.dataset)

### Training Loop

In [ ]:
history = []
for epoch in range(EPOCHS):
    train_loss = train_epoch(model, train_loader, optimizer)
    val_loss = estimate_loss(model, val_loader)
    history.append(val_loss)
    print(f"Epoch {epoch + 1:2d} | train loss {train_loss:.4f} | val loss {val_loss:.4f}")

### Visualize Training

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(range(1, EPOCHS + 1), history, marker='o')
plt.xlabel('Epoch')
plt.ylabel('Validation loss')
plt.title('TinyGPT training')
plt.tight_layout()
plt.show()

## Generate Text

In [ ]:
def generate_text(model, start, length, temperature=0.8):
    """
    Generate readable text from a seed string.

    Parameters:
        model (nn.Module): Trained model.
        start (str): Seed text.
        length (int): Number of characters to generate.
        temperature (float): Sampling temperature.

    Returns:
        str: The generated text.
    """
    model.eval()
    idx = torch.tensor([[STOI[ch] for ch in start]], device=DEVICE)
    with torch.no_grad():
        out = model.generate(idx, length)
    return ''.join(ITOS[i] for i in out[0].tolist())

In [ ]:
print(generate_text(model, 'ROMEO: ', 400))

## Save Model

In [ ]:
torch.save(model.state_dict(), 'tinygpt_shakespeare.pt')
print('saved tinygpt_shakespeare.pt')

## Load Model

In [ ]:
loaded_model = TinyGPT(VOCAB_SIZE, EMBED_DIM, N_HEADS, N_LAYERS, BLOCK_SIZE, FF_DIM).to(DEVICE)
loaded_model.load_state_dict(torch.load('tinygpt_shakespeare.pt', map_location=DEVICE))
loaded_model.eval()
print('loaded tinygpt_shakespeare.pt')

## Inference

In [ ]:
print(generate_text(loaded_model, 'JULIET: ', 400))